# Project 2

Alexander Matteson

GitHub link: https://github.com/AlexanderMatteson/Project2_ee513_Matteson

In [1]:
# imports:
import glob, os
import numpy as np
import rawpy

### part A: the classical pipeline


1. Black level subtraction:

In [2]:
folder = r"C:\Users\ammat\Documents\ee513\Project2\DarkFields"
files = sorted(set(glob.glob(os.path.join(folder, "*.dng")) + glob.glob(os.path.join(folder, "*.DNG"))))
assert len(files) > 0, "No .dng files found"
print(f"Found {len(files)} files")

def channel_planes(raw):
    """Return ({name: 2D array}, {name: manufacturer black}) for each CFA channel."""
    img = raw.raw_image_visible.astype(np.float64)
    pat = raw.raw_pattern
    desc = raw.color_desc.decode()
    bl = raw.black_level_per_channel
    planes, mfr = {}, {}
    g_seen = 0
    for y in range(2):
        for x in range(2):
            idx = pat[y, x]
            name = desc[idx]
            if name == "G":
                g_seen += 1
                name = f"G{g_seen}"
            planes[name] = img[y::2, x::2]
            mfr[name] = bl[idx]
    return planes, mfr

# ---- Load frames ----
frames, mfr_black, white = [], [], []
for f in files:
    with rawpy.imread(f) as raw:
        planes, mfr = channel_planes(raw)
        frames.append(planes)
        mfr_black.append(mfr)
        white.append(raw.white_level)
        print(f"{os.path.basename(f)}: shape={raw.raw_image_visible.shape}, "
              f"pattern={raw.color_desc.decode()}, white_level={raw.white_level}, "
              f"manufacturer black per channel={list(raw.black_level_per_channel)}")

channels = list(frames[0].keys())

# ---- Per-frame stats ----
print("\n=== Per-frame results ===")
for i, f in enumerate(files):
    print(os.path.basename(f))
    for c in channels:
        v = frames[i][c]
        print(f"  {c:>2}: mean={v.mean():9.3f}  median={np.median(v):9.3f}  std={v.std():7.3f}")

# ---- Pooled stats vs manufacturer ----
print("\n=== Pooled across all frames ===")
print(f"{'ch':>3} {'mean':>10} {'median':>10} {'std':>8} {'manufacturer':>13} {'diff(mean-mfr)':>15}")
for c in channels:
    v = np.concatenate([fr[c].ravel() for fr in frames])
    mfr = np.mean([m[c] for m in mfr_black])
    print(f"{c:>3} {v.mean():10.3f} {np.median(v):10.3f} {v.std():8.3f} {mfr:13.3f} {v.mean()-mfr:15.3f}")

if len(set(white)) == 1:
    print(f"\nWhite level (manufacturer): {white[0]}")

Found 3 files
RawCam-2026-11-11-164841.dng: shape=(3024, 4032), pattern=RGBG, white_level=4095, manufacturer black per channel=[528, 528, 528, 528]
RawCam-2026-11-11-164919.dng: shape=(3024, 4032), pattern=RGBG, white_level=4095, manufacturer black per channel=[528, 528, 528, 528]
RawCam-2026-11-11-165101.dng: shape=(3024, 4032), pattern=RGBG, white_level=4095, manufacturer black per channel=[528, 528, 528, 528]

=== Per-frame results ===
RawCam-2026-11-11-164841.dng
   R: mean=  527.909  median=  528.000  std=  6.801
  G1: mean=  528.005  median=  528.000  std=  6.001
  G2: mean=  528.005  median=  528.000  std=  5.373
   B: mean=  527.796  median=  528.000  std=  6.458
RawCam-2026-11-11-164919.dng
   R: mean=  527.777  median=  528.000  std=  6.772
  G1: mean=  527.815  median=  528.000  std=  5.958
  G2: mean=  527.816  median=  528.000  std=  5.334
   B: mean=  527.724  median=  528.000  std=  6.453
RawCam-2026-11-11-165101.dng
   R: mean=  527.952  median=  528.000  std=  6.782
  

In [3]:
blackR = 527.879
blackG = 528.015  # both G1 and G2 had the same values
blackB = 527.779
b_list = [blackR, blackG, blackB]

def subtrackBlackLevel(img, b_list, pattern="RGGB", clip=False):
    """
    Subtract the per-channel black level from a Bayer mosaic.

    img     : 2D raw mosaic array (H x W)
    b_list  : [blackR, blackG, blackB]
    pattern : 2x2 CFA layout read row by row, top-left first,
              e.g. "RGGB", "BGGR", "GRBG", "GBRG"
    clip    : if True, clamp negatives to 0 after subtraction.
              Default False keeps signed values, which preserves
              noise statistics in the shadows.
    Returns a float32 mosaic of the same shape.
    """
    levels = {"R": b_list[0], "G": b_list[1], "B": b_list[2]}
    out = img.astype(np.float32)  # avoid uint16 wraparound

    # positions in the 2x2 tile: (row offset, col offset)
    positions = [(0, 0), (0, 1), (1, 0), (1, 1)]
    for (y, x), ch in zip(positions, pattern.upper()):
        out[y::2, x::2] -= levels[ch]

    if clip:
        np.maximum(out, 0, out=out)
    return out

2. Lens Shading: